## Proyecto Final: Sistema de Recomendación E-commerce
### Modelo 1: Filtrado colaborativo

**Equipo:** MetricEdge

**Autor:** Sara Henao

**Dataset:** interacciones_clean.csv

Fuente de origen: E-commerce Sales & Customer Analytics (150k), Kaggle

### Objetivo de este notebook
Entrenamiento del primer modelo por filtrado colaborativo, basado en el principio "los usuarios que tuvieron comportamientos parecidos en el pasado, probablemente compartirán preferencias en el futuro"

**Importante:** A diferencia de los métodos basados en contenido (que se apoyan en las características de los productos o en el perfil del usuario), el filtrado colaborativo aprende directamente del comportamiento colectivo: compras, calificaciones, clics o reproducciones. 

In [9]:
#Importar librerías
import pandas as pd
import numpy as np
from implicit.als import AlternatingLeastSquares
from implicit.evaluation import train_test_split, precision_at_k
import scipy.sparse as sparse

import warnings
warnings.filterwarnings('ignore')

In [4]:
#Cargar datos customer_clean.csv, interacciones_clean.csv, order_items_clean.csv, product_clean.csv, sales_clean.csv

df_interactions = pd.read_csv('../data/processed/interacciones_clean.csv')

display(df_interactions.head())


,order_id,product_id,quantity,unit_price,discount_percentage,discount_amount,gross_sales,tax_amount,shipping_cost,net_sales,product_cost,profit,customer_id,order_date,order_status
0,ORD-100004,PROD-000475,1,12.91,0.094500,1.22,12.91,2.10,10.10,23.89,4.26,9.53,CUST-022489,2025-03-29,Completed
1,ORD-100004,PROD-000782,1,116.11,0.098958,11.49,116.11,18.83,2.30,125.75,67.97,55.48,CUST-022489,2025-03-29,Completed
2,ORD-100004,PROD-000912,1,335.86,0.042012,14.11,335.86,57.92,4.06,383.73,195.75,183.92,CUST-022489,2025-03-29,Completed
3,ORD-100016,PROD-000319,2,176.23,0.180701,63.69,352.46,20.21,3.80,312.78,157.12,151.86,CUST-011235,2023-03-07,Completed
4,ORD-100016,PROD-001037,1,61.00,0.090164,5.50,61.00,3.88,15.15,74.53,24.60,34.78,CUST-011235,2023-03-07,Completed


**Concepto de entrenamiento:**

**Datos de Entrada:** Solo ID de Usuario, ID de Producto e Interacción (cantidad de cada producto comprado o rating de producto)

**¿Qué aprende?** Similitudes o Factores latentes (gustos abstractos)

**Mecanismo de "Entrenamiento":** Factorización de matrices SVD(Singular Value Decomposition)/ALS(Alternating Least Squares) o cálculo de distancias (Coseno) KNN

Para este modelo se escoge factorización de matrices, ya que tiene la ventaja de ser escalable a millones de usuarios y productos sin depender de información textual, permite descubrir relaciones complejas que el método del coseno ignora y tiene un mejor manejo de de la dispersión (sparsity) al capturar las tendencias generales, incluyendo a los articulos de venta y de nicho (long tail). Además, se selecciona el algoritmo ALS optimizado para datos implícitos. La elección se justifica debido a que el dataset en la tabla *interactions* cuenta con más de 755000 transacciones, donde la interacción clave es el comportamiento/compra (quantity) y no de calificaciones individuales de productos(rating directos por el usuario). Este último dato que no se tiene actualmente, lo que se tiene es un rating global por producto, lo que no permite rastrear la satisfacción del usuario individual y por ende limitando la aplicación de un algoritmo como SVD, que normalmente se usa cuando el objetivo del negocio es predecir la satisfacción exacta (calificación) del cliente.

Otro factor a tener en cuenta es el comportamiento matemático de los algoritmos. Al trabajar con una matriz pivotada, se generarán vacíos, aquí es donde difieren las interpretaciones de ALS y SVD. ALS toma esos vacíos generados y los transforma en "0", interpretandolos como un desinterés potencial o falta de exposición del artículo, en cambio SVD
toma esos NaN, los ignora y solo entrena con las celdas que sí tienen datos, ya que fue diseñado para datos explícitos (estrellas, reviews, ratings,etc)


El valor del filtrado colaborativo radica en su capacidad de aprender gustos implícitos sin conocer las características del producto, lo que lo hace aplicable en contextos donde los metadatos son incompletos o heterogéneos

In [5]:
#Dataframe con las columnas necesarias para el modelo 'customer_id','product_id','quantity'
df_model = df_interactions[['customer_id','product_id','quantity']].copy()

#consolidar si un cliente  compró el mismo producto en diferentes órdenes
df_model = df_model.groupby(['customer_id','product_id'])['quantity'].sum().reset_index()

print(f"Registros únicos tras la consolidación: {df_model.shape[0]}")

Registros únicos tras la consolidación: 352031


**Matriz dispersa**

La librería implicit no entiende los IDs de clientes o productos si estos son cadenas de texto (CUST_10293 o PROD_ABC). Necesita que se conviertan a números enteros indexados (0, 1, 2, 3...). Además, al hacer una tabla pivote tradicional con .pivot(), Python intentaría guardar millones de ceros en la memoria RAM. Para evitarlo, se utiliza una Matriz Dispersa en formato CSR (Compressed Sparse Row), la cual solo almacena en memoria las celdas que sí tienen datos

In [6]:
# 1. Convertir IDs de texto a categorías numéricas (Factores/Categorías)
df_model['customer_id'] = df_model['customer_id'].astype("category")
df_model['product_id'] = df_model['product_id'].astype("category")

# 2. Crear columnas con los códigos numéricos indexados
df_model['user_code'] = df_model['customer_id'].cat.codes
df_model['item_code'] = df_model['product_id'].cat.codes

# 3. Guardar diccionarios de mapeo para poder recuperar los IDs reales una vez el modelo esté entrenado y recomiende
# (Mapeo de código numérico -> ID original de Kaggle)
user_map = dict(enumerate(df_model['customer_id'].cat.categories))
item_map = dict(enumerate(df_model['product_id'].cat.categories))

# 4. Crear la Matriz Dispersa en formato CSR
# Sintaxis de SciPy: csr_matrix((datos, (filas, columnas)))
user_item_matrix = sparse.csr_matrix(
    (df_model['quantity'].astype(float), (df_model['user_code'], df_model['item_code']))
)

print(f"Dimensiones de la matriz (Usuarios x Productos): {user_item_matrix.shape}")
print(f"Porcentaje de densidad (celdas llenas): {user_item_matrix.nnz / (user_item_matrix.shape[0] * user_item_matrix.shape[1]) * 100:.4f}%")


Dimensiones de la matriz (Usuarios x Productos): (24838, 1175)
Porcentaje de densidad (celdas llenas): 1.2062%


**Entrenamiento  y prueba del modelo**

Para evaluar el modelo de diltrado colaorativo ALS no se puede realizar un train_test_split como se realiza en otros modelos, se debe hacer una división basada en ocultar interacciones o máscara/

Ajustes de calibración del modelo
- La penalización de la "Falta de Ceros" en Quantity: Por defecto, el algoritmo ALS asume que si el valor de compra es 1, la confianza es baja, y si es 10, es alta. En el dataset la gran mayoría de los clientes compraron solo 1 unidad de un producto, como se observó en el EDA primario, para ALS esa señal es casi tan débil como un cero.

Por este motivo se debe escalar la matriz con un factor alfa (α) para que el modelo reintreprete que "si hay una compra, aumenta drásticamente mi confianza de que le gusta". El estándar es multiplicar la matriz de entrenamiento por un valor entre 15 y 40.

In [10]:
#Dividir la matriz dispersa  de forma correcta para isstems de recomendación
#El 20% de las interacciones se ocultan para hacer la validación posterior
train_matrix, test_matrix = train_test_split(user_item_matrix, train_percentage=0.8, random_state=42)

# Multiplicamos la matriz de entrenamiento por un factor alfa (ej. 40)
# Esto le indica al algoritmo que una compra de cantidad 1 SÍ es una señal fuerte.
alpha_val = 40
train_matrix_scaled = train_matrix.multiply(alpha_val).astype('float32') #(float32)Requerimiento de implicit para que funcione

#Configurar el modelo ALS
model_ALS = AlternatingLeastSquares(
    factors=64,          # Número de factores latentes (vectores ocultos) estándar entre 64 y 128. Muy bajo|underfitting
                            #muy alto| overffiting
    regularization=0.1,  # Penalización para evitar sobreajuste (overfitting). Estandar 0.01, 0.1, 1.0. 
                            #si falla drásticamente al recomendar cosas nuevas, debes subir este valor 
    iterations=20,       # Cuántas iteraciones hará para ajustar las matrices de usuarios y productos
                            #entre 15 y 30 iteraciones son suficientes para que el modelo converja
    random_state=42      # Semilla para que los resultados sean reproducibles
)

# 2. Entrenar el modelo
model_ALS.fit(train_matrix_scaled)

#Evaluar el modelo con métrica de E-commerce precision@k
# Precision@K mide: De los top 10 productos que le recomendamos al usuario, ¿cuántos compró realmente en el test_matrix?
#Forzar formato CSR explícito en la evaluación
p_at_k = precision_at_k(
    model= model_ALS, 
    train_user_items= train_matrix_scaled.tocsr(),
    test_user_items= test_matrix.tocsr(),
    K=10,
    show_progress=True
)

print(f"\nLa Precisión@10 del modelo es: {p_at_k * 100:.2f}%")


100%|██████████| 22276/22276 [00:04<00:00, 4889.22it/s]


La Precisión@10 del modelo es: 0.86%


In [11]:
#Bloque de auditoría
# 1. Contar cuántos productos únicos tiene cada cliente
productos_por_cliente = df_model.groupby('customer_id')['product_id'].count()

# 2. Calcular cuántos clientes tienen exactamente 1 solo producto comprado
clientes_con_un_solo_producto = (productos_por_cliente == 1).sum()
total_clientes = len(productos_por_cliente)
porcentaje_un_producto = (clientes_con_un_solo_producto / total_clientes) * 100

print(f"Total de clientes en el dataset: {total_clientes}")
print(f"Clientes que solo compraron 1 único producto: {clientes_con_un_solo_producto}")
print(f"Porcentaje de clientes con solo 1 producto: {porcentaje_un_producto:.2f}%\n")

# 3. Ver una distribución rápida (Percentiles)
print("Distribución del número de productos comprados por cliente:")
print(productos_por_cliente.describe(percentiles=[0.25, 0.5, 0.75, 0.90]))

Total de clientes en el dataset: 24838
Clientes que solo compraron 1 único producto: 156
Porcentaje de clientes con solo 1 producto: 0.63%

Distribución del número de productos comprados por cliente:
count    24838.000000
mean        14.173082
std          7.034781
min          1.000000
25%          9.000000
50%         13.000000
75%         18.000000
90%         24.000000
max         63.000000
Name: product_id, dtype: float64


El resultado de 0.86% no es un error de código, sino una limitación matemática de la métrica sobre este dataset debido a tres factores:
- Cada cliente compra una media de 14 productos. Al ocultar el 20% para el test, solo quedan 2 o 3 productos objetivo por adivinar. Como la métrica exige recomendar 10 artículos, la precisión máxima por usuario está topada a un 3/10 = 30 aunque el modelo sea perfecto. Al promediar esto entre más de 22,000 usuarios, el porcentaje global baja drásticamente.
- Inmensidad del Catálogo: El algoritmo debe seleccionar solo 10 artículos dentro de un mar de 1,175 productos disponibles. Acertar el ítem exacto del test es estadísticamente muy difícil.
- Rigidez Binaria: La métrica evalúa como fracaso rotundo (0) si recomiendas algo lógico que al cliente le encantaría, pero que simplemente no compró aún dentro de la ventana de pruebas.